# Physics branch: train and compare (T4 GPU, about 2 h)

**What.** The same model as Part B of `colab_fskpsk_fix.ipynb` (seed 4, n_fft 64, expert features, spectrum branch, 15 epochs, fixed data `f993d8ebc417`) **plus the physics branch**: 56 measurements per window that separate signals the CNN mixes up (PSK order, analog vs tones, APSK vs band noise, OFDM echo, on/off timing). Why and the evidence: `docs/SEPARABILITY_ANALYSIS.md` section 6.1 (fair check +1.1 top-1, +1.4 at >= 10 dB).

**Account.** Sign in to Colab as **evan17tan** (owns FOI26, so it is at `MyDrive/FOI26`) and pick the same account in the Drive pop-up. Another account works if it has edit access and a shortcut to FOI26 in My Drive.

**Runtime.** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.

**If Colab disconnects:** Run all again. Training resumes from the last saved epoch (same run name).

**Result.** `FOI26/results/2026-11-07_physics_seed4/`: `best.pt`, `scorecard.json`, `calibration.json`, `compare.json`. Keep the physics model only if it beats the run without it on the same test set (step 6).

## 1. Settings

In [ ]:
SEED = 4
BRANCH = 'eavan-accuracy-plan'                    # Eileen's sed-39 work + the physics branch
RUN_NAME = '2026-11-07_physics'
EPOCHS = 15
XF = '--expert-features --full-spectrum --physics-features'
DATA_NAME = 'combined_data_labels_v2_fskpskfix'   # f993d8ebc417, the right data (docs/DATA_VERSIONS.md)

# Models to compare with on the same test set (skipped if not on Drive yet)
COMPARE = {
    'no physics (Part B, fixed data)': 'results/2026-11-07_nfft64_ep15_labelsv2fix_expert_spectrum_seed4/best.pt',
    'Eileen spectrum (old data, 52%)': 'results/2026-11-06_nfft64_ep15_labelsv2_expert_spectrum_seed4/best.pt',
}

## 2. Drive (finds FOI26 in the signed-in account)

In [ ]:
import glob, json, os
from google.colab import drive
drive.mount('/content/drive')
found = [p for p in ['/content/drive/MyDrive/FOI26', '/content/drive/MyDrive/sedic/FOI26']
         + sorted(glob.glob('/content/drive/MyDrive/*/FOI26')) + sorted(glob.glob('/content/drive/Shareddrives/*/FOI26'))
         if os.path.isdir(p)]
assert found, 'FOI26 not found: sign in as evan17tan, or add a shortcut to FOI26 in My Drive'
FOI26 = found[0]
DATA = f'{FOI26}/{DATA_NAME}'
CKPT_DIR = f'{FOI26}/results/{RUN_NAME}_seed{SEED}'
assert os.path.exists(f'{DATA}/combined.json'), f'fixed data missing: {DATA}'
os.makedirs(CKPT_DIR, exist_ok=True)
print('FOI26   ', FOI26)
print('data    ', DATA, json.load(open(f'{DATA}/combined.json'))['combined_version'])
print('saves to', CKPT_DIR)

## 3. Code and GPU

In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log -1 --format='commit %h  %s'
!pip install -q pyyaml
import torch
assert torch.cuda.is_available(), 'needs a GPU: Runtime -> Change runtime type -> T4 GPU, then Run all'
print('GPU:', torch.cuda.get_device_name(0))

## 4. Copy the data to the Colab disk (about 5 min) and a 1-minute check

In [ ]:
if not os.path.exists('/content/combined/combined.json'):
    !mkdir -p /content/combined
    !cp -r "{DATA}/train" "{DATA}/val" "{DATA}/test" /content/combined/
    !cp "{DATA}/combined.json" /content/combined/
print('local copy:', json.load(open('/content/combined/combined.json'))['combined_version'])
!python scripts/train_week1.py --seed {SEED} --data /content/combined --limit 2000 --epochs 1 --batch-size 256 {XF} --ckpt-dir /content/try --fresh

## 5. Train (about 2 h) and calibrate
One line per epoch; every epoch is saved to Drive; the test split is scored at the end.

In [ ]:
!python scripts/train_week1.py --seed {SEED} --data /content/combined --epochs {EPOCHS} --batch-size 256 {XF} --ckpt-dir "{CKPT_DIR}" --eval-test
!python scripts/calibrate_week1.py --ckpt "{CKPT_DIR}/best.pt" --data /content/combined --cache /content/logits_physics

## 6. Compare on the same test set (fixed data): overall, >= 10 dB, per family

In [ ]:
import numpy as np
from src.config import CLASS_GROUPS, DATASET_CLASSES as C
from src.models.amc_cnn import AMC_CNN
dev = torch.device('cuda')
FAM = {c: g for g, cs in CLASS_GROUPS.items() for c in cs}; FAM['NOISE_FLOOR'] = 'NOISE'

def load(path):
    ck = torch.load(path, map_location=dev, weights_only=False); m = ck['meta']
    net = AMC_CNN(num_classes=len(C), input_len=m['input_len'], stft_n_fft=m['stft_n_fft'], **m.get('model_flags', {})).to(dev)
    net.load_state_dict(ck['state_dict'])
    return net.eval()

@torch.no_grad()
def top_of(net, X):
    return np.concatenate([net(torch.from_numpy(np.asarray(X[s:s + 1024], dtype=np.float32)).to(dev)).argmax(1).cpu().numpy()
                           for s in range(0, len(X), 1024)])

models = {'physics (this run)': f'{CKPT_DIR}/best.pt'}
for name, rel in COMPARE.items():
    if os.path.exists(f'{FOI26}/{rel}'): models[name] = f'{FOI26}/{rel}'
    else: print('not on Drive yet, skipped:', name)
X = np.load('/content/combined/test/X.npy', mmap_mode='r')
y = np.load('/content/combined/test/y.npy').astype(bool); snr = np.load('/content/combined/test/snr_labels.npy')
tf = np.array([FAM[C[i]] for i in y.argmax(1)]); single = y.sum(1) == 1
fams = sorted(set(FAM.values()))
res = {}
for name, path in models.items():
    hit = y[np.arange(len(y)), top_of(load(path), X)]
    res[name] = {'top1': float(hit.mean()), 'top1_high': float(hit[snr >= 10].mean()),
                 'family': {g: float(hit[single & (tf == g)].mean()) for g in fams}}
cols = list(res)
print(f"{'':14s}" + ''.join(f'{c[:30]:>32s}' for c in cols))
print(f"{'top-1':14s}" + ''.join(f"{res[c]['top1']:32.3f}" for c in cols))
print(f"{'top-1 >=10 dB':14s}" + ''.join(f"{res[c]['top1_high']:32.3f}" for c in cols))
for g in fams:
    print(f'{g:14s}' + ''.join(f"{res[c]['family'][g]:32.3f}" for c in cols))
json.dump(res, open(f'{CKPT_DIR}/compare.json', 'w'), indent=2)
print('wrote', f'{CKPT_DIR}/compare.json')

## 7. Done
Tell Eavan: `FOI26/results/2026-11-07_physics_seed4` has everything. To use it in the pipeline, copy `best.pt` to `results/classifier/` on the PC (it runs in PyTorch; this model does not export to ONNX yet).